# 00. Environment and software versions

Records where the notebooks were executed and which software versions produced the results, so that every number in this repository can be traced to a known setup.

Two machines are involved:

- **search machine**: runs FragPipe on the raw files (Windows desktop, Intel i5-13500, 14 cores / 20 threads, 32 GB RAM, SSD); its tool versions are read from the FragPipe log;
- **notebook environment**: runs the Python notebooks; described by this notebook wherever it is executed.

Outputs: `results/environment_notebooks.json`, `results/software_versions.tsv`.

In [3]:
import importlib, json, platform, re, shutil, sys
from pathlib import Path
import pandas as pd
import psutil

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
RES = ROOT / "results"; RES.mkdir(exist_ok=True)

env = {
    "os": f"{platform.system()} {platform.release()}",
    "machine": platform.machine(),
    "physical_cores": psutil.cpu_count(logical=False),
    "logical_cores": psutil.cpu_count(logical=True),
    "ram_gb": round(psutil.virtual_memory().total / 1e9, 1),
    "python": sys.version.split()[0],
    "project_disk_free_gb": round(shutil.disk_usage(ROOT).free / 1e9),
}
(RES / "environment_notebooks.json").write_text(json.dumps(env, indent=2), encoding="utf-8")
env

{'os': 'Windows 10',
 'machine': 'AMD64',
 'physical_cores': 14,
 'logical_cores': 20,
 'ram_gb': 34.0,
 'python': '3.10.22',
 'project_disk_free_gb': 138}

In [4]:
PACKAGES = ["numpy", "pandas", "scipy", "statsmodels", "matplotlib", "openpyxl", "requests", "tqdm", "psutil", "nbformat", "nbclient", "ipykernel"]
rows = [("notebooks", "python", sys.version.split()[0])]
for name in PACKAGES:
    try:
        rows.append(("notebooks", name, importlib.metadata.version(name)))
    except importlib.metadata.PackageNotFoundError:
        rows.append(("notebooks", name, "not installed"))

log = ROOT / "results_large" / "search_ALL_SETS" / "run_log.txt"
if log.exists():
    text = log.read_text(encoding="utf-8", errors="replace")
    block = text[text.index("Version info:"):][:1500]
    for tool, version in re.findall(r"^(.+?) version (\S+)$", block, flags=re.M):
        if tool in ("FragPipe", "MSFragger", "MSBooster", "Percolator", "Philosopher", "IonQuant", "TMT-Integrator"):
            rows.append(("search", tool, version))
    m = re.search(r"Java Info: ([^,\n]+)", text)
    if m: rows.append(("search", "Java", m.group(1)))
else:                                   # versions of the run reported in this repository
    rows += [("search", t, v) for t, v in [("FragPipe", "24.0"), ("MSFragger", "4.4.1"), ("MSBooster", "1.4.14"), ("Percolator", "3.7.1"),
                                           ("Philosopher", "5.1.3-RC9"), ("IonQuant", "1.11.20"), ("TMT-Integrator", "6.1.3")]]
rows.append(("spectrum extraction", "ThermoRawFileParser", "2.0.0-dev"))
versions = pd.DataFrame(rows, columns=["step", "software", "version"])
versions.to_csv(RES / "software_versions.tsv", sep="\t", index=False)
versions

,step,software,version
0,notebooks,python,3.10.22
1,notebooks,numpy,2.2.6
2,notebooks,pandas,2.3.3
3,notebooks,scipy,1.15.3
4,notebooks,statsmodels,0.15.0
5,notebooks,matplotlib,3.10.9
6,notebooks,openpyxl,3.1.5
7,notebooks,requests,2.34.2
8,notebooks,tqdm,4.70.1
9,notebooks,psutil,7.2.2


## Requirements for repeating the search

| Resource | Needed | Why |
|---|---|---|
| Free disk | about 300 GB | 96 Thermo `.raw` files (200 GB) plus intermediate and result files |
| RAM | 32 GB | MSFragger index for the variant-aware human database with TMT modifications (run with 24 GB) |
| Time | about 3 hours | 2 h 26 min search and validation, 28 min quantification, on the search machine above |

The notebooks themselves need little: 4 GB of RAM is enough.